In [1]:
import argparse
import os
import pickle
import pprint
import csv

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "2.2.4":
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:

from kawada_env_cnoid import KawadaBaseEnvChoreonoid as RL_Env

In [3]:

# from bex24_env_cnoid import RLEnvChoreonoid as RL_Env

In [4]:
# exp_name = 'kawada-walking-1001'
# ckpt = 1000

In [5]:
exp_name = 'ishiki-walking-no-vel'
ckpt = 2000

In [6]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}
env_cfg["rotorInertia"] = 0.1

In [7]:
env = RL_Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        # dt=env_cfg['dt'],
        dt=0.01,
        # substeps=env_cfg['substeps'],
        substeps=5,
        show_viewer=True,
    )

In [8]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)


/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [9]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [10]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)
    print("2 : ", actions)
    obs, rews, dones, infos = env.step(actions)
    print(obs)
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(obs.cpu().numpy().flatten())
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")

cnt : 0
2 :  tensor([[ 1.2061,  1.8438, -0.4992,  2.1252, -1.8702,  1.0577,  0.2597,  1.6540,
         -0.4176, -0.0076, -1.8781,  1.0838]], device='cuda:0')


/userdir/samples/../irsl_rl/rl_env_base.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/samples/../irsl_rl/rl_env_cnoid.py:84: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  self.dof_pos = torch.tensor([sbody.angleVector()]).to(torch.float32).to(self.device)


tensor([[-5.6001e-06, -8.6880e-03, -2.1613e-06,  2.0291e-09,  3.2152e-20,
         -1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  4.1678e-08,
          8.2047e-08, -3.1018e-04,  7.9763e-04, -4.1813e-04, -9.9951e-08,
         -2.2947e-07, -1.2626e-07, -3.1036e-04,  7.9787e-04, -4.1819e-04,
          2.0328e-06,  1.0419e-06,  2.0512e-06, -7.7551e-03,  1.9941e-02,
         -1.0448e-02, -2.4988e-06, -5.7368e-06, -3.1565e-06, -7.7598e-03,
          1.9948e-02, -1.0450e-02,  5.0819e-05,  1.2061e+00,  1.8438e+00,
         -4.9916e-01,  2.1252e+00, -1.8702e+00,  1.0577e+00,  2.5969e-01,
          1.6540e+00, -4.1757e-01, -7.5878e-03, -1.8781e+00,  1.0838e+00]],
       device='cuda:0')
torques: [ 1.59160299e-13 -7.29921419e-14  1.67400678e-03 -2.90351253e-03
  5.43352198e-02  2.17351506e-12 -6.86668107e-14  5.04415353e-15
  1.67400678e-03 -2.90351253e-03  5.43352198e-02 -5.47664442e-13]
データ収集: step 1


In [11]:
# 既存のforループを置き換え
num_steps = 20
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        torques = env.sim.sbody.getTorques()
        obs, rews, dones, infos = env.step(actions)
        
        # データを記録
        step_data.append(cnt)
        # action_data.append(actions.cpu().numpy().flatten())
        obs_data.append(obs.cpu().numpy().flatten())
        torque_data.append(torques.copy())
        
        if i % 20 == 0:  # 20ステップごとに進行状況を表示
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected")

Step 1/20, Total steps: 1
データ収集完了: 20 steps collected


In [12]:
# for i in range(500):
#     obs, _ = env.reset()
#     with torch.no_grad():
#         actions = policy(obs)
#         obs, rews, dones, infos = env.step(actions)

In [13]:
env.sim.stop()

In [14]:

# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_simple.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_ishiki-walking-no-vel_ckpt2000_simple.csv
データ形状: (21, 58)
